In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)

from pathlib import Path

BASE_DIR = Path.cwd().parent

clean_transactions = BASE_DIR / "resources" / "clean_mpesa_transactions.csv"

df = pd.read_csv(clean_transactions)

In [2]:
print("Columns:", df.columns.tolist())
print("Number of transactions:", len(df))
print("Label counts:", df['Category'].value_counts())

Columns: ['Receipt No', 'Completion Time', 'Details', 'Transaction Status', 'Paid In', 'Withdrawn', 'Balance', 'Notes', 'Year', 'Month', 'Date', 'Day', 'Amount', 'Category']
Number of transactions: 4081
Label counts: Category
Expense    3548
Income      360
Neutral     173
Name: count, dtype: int64


## Clustering to reveal transaction patterns

In [3]:
from sklearn.cluster import DBSCAN
from sentence_transformers import SentenceTransformer

# Filling NaNs with empty strings
details_texts = df['Details'].fillna("").tolist()

model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight embedding model
embeddings = model.encode(details_texts)

print(embeddings[:5])


[[-0.0518424   0.05095716 -0.00100883 ...  0.02403557 -0.05793547
  -0.03286178]
 [-0.04160628  0.06078717  0.0122063  ... -0.04113406  0.01323489
  -0.06800348]
 [-0.04161704  0.07661207 -0.0026605  ... -0.05808166 -0.01867538
  -0.05602515]
 [-0.04161704  0.07661207 -0.0026605  ... -0.05808166 -0.01867538
  -0.05602515]
 [-0.0518424   0.05095716 -0.00100883 ...  0.02403557 -0.05793547
  -0.03286178]]


In [4]:
clustering = DBSCAN(eps=0.2, min_samples=1, metric='cosine').fit(embeddings)
df['Cluster'] = clustering.labels_

In [5]:
# Grouping by cluster to inspect patterns
clusters = df.groupby('Cluster')['Details'].apply(list)
sorted_clusters = clusters.sort_values(key=lambda x: x.map(len), ascending=False)

In [ ]:
print("Clustered Patterns:")
for cluster_id, messages in sorted_clusters.items():
    print(f"\nCluster {cluster_id} (n={len(messages)}):")
    for msg in messages[:5]:  # showing first 5 messages per cluster
        print(f"  {msg}")

### Classification Stage 1: Regex

In [7]:
import re

def classify_with_regex(text):
    if pd.isna(text):
        return None
        
    text = text.lower()

    patterns = [
        # Transaction costs (first to avoid misclassification)
        (r"customer transfer of funds charge", "Transaction Costs (Send Money)"),
        (r"pay merchant charge", "Transaction Costs (Pochi la Biashara)"),
        (r"pay bill charge", "Transaction Costs (Paybill)"),
        (r"withdrawal charge", "Transaction Costs (Withdraw)"),

        # Deposits / withdrawals
        (r"deposit of funds at agent till", "M-PESA Deposits"),
        (r"m-shwari deposit", "M-Shwari Deposits"),
        (r"customer withdrawal at agent till", "MPESA Withdrawals"),
        (r"m-shwari withdraw", "M-Shwari Withdrawals"),

        # Incoming
        (r"funds received from", "Received (Send Money)"),
        (r"business payment from", "Received (Bank)"),
        (r"offnet b2c transfer", "Received (Send Money)"),

        # Outgoing
        (r"offnet c2b transfer", "Sent"),
        (r"customer transfer to", "Sent"),

        # Shopping
        (r"merchant payment", "Shopping (Till)"),
        (r"customer payment to small business", "Shopping (Pochi la Biashara)"),

        # Bills
        (r"pay bill online", "Bills (Online)"),
        (r"pay bill to", "Bills (Paybill)"),
        (r"kplc prepaid", "Bills (Electricity)"),
        (r"bundle purchase|customer bundle purchase|safaricom data bundles", "Bills (Data Bundles)"),
        (r"airtime purchase", "Bills (Airtime Purchase)"),

        # Reversals
        (r"reversal", "Reversals"),
    ]

    for pattern, label in patterns:
        if re.search(pattern, text):
            return label

    return None
    
df["Subcategory"] = df["Details"].apply(lambda x: classify_with_regex(x))

In [8]:
# Transactions not classified by the regex rules
ml_data = df[df["Subcategory"].isna()]
ml_data.shape

(169, 16)

## LLM fallback

In [9]:
from groq import Groq
import os

from dotenv import load_dotenv
load_dotenv()

groq = Groq()

def classify_transactions_batch(text_list):
    """
    text_list: list of transaction strings
    Returns: list of categories in the same order
    """
    # Build prompt for all transactions at once
    prompt = "You are a financial transaction classifier. Classify each transaction into one of the following categories:\n\n"
    prompt += "M-PESA Deposits, M-Shwari Deposits, MPESA Withdrawals, M-Shwari Withdrawals, Received (Send Money), Received (Bank), Sent (Send Money), Shopping (Till), Shopping (Pochi la Biashara),Bills (Online), Bills (Paybill), Bills (Electricity), Bills (Data Bundles), Bills (Airtime Purchase), Transaction Costs (Send Money), Transaction Costs (Paybill), Transaction Costs (Withdraw), Reversals, Unclassified.\n\n"
    prompt += "Return the category for each transaction on a separate line using <category></category> tags. Do NOT include explanations.\n\n"
    prompt += "Transactions:\n"
    for i, text in enumerate(text_list, 1):
        prompt += f"{i}. {text}\n"

    chat_completion = groq.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.3-70b-versatile",
        temperature=0.3
    )

    content = chat_completion.choices[0].message.content

    valid_categories = {"M-PESA Deposits", "M-Shwari Deposits", "MPESA Withdrawals", "Received (Send Money)", "Received (Bank)", "Sent", "Shopping (Till)", "Shopping (Pochi la Biashara)", "Bills (Online)", "Bills (Paybill)", "Bills (Electricity)", "Bills (Data Bundles)", "Bills (Airtime Purchase)", "Transaction Costs (Send Money)", "Transaction Costs (Paybill)", "Transaction Costs (Withdraw)", "Reversals", "Unclassified"}

    # Extraction
    categories = []

    for match in re.findall(r'<category>(.*?)<\/category>', content, flags=re.DOTALL):
        cat = match.strip()
        if cat not in valid_categories:
            cat = "Unclassified"
        categories.append(cat)
        
    # If the number of categories returned < number of transactions, pad with "Unclassified"
    while len(categories) < len(text_list):
        categories.append("Unclassified")

    return categories

## Classify transactions using batch LLM + regex fallback

In [11]:
llm_needed_mask = df["Details"].apply(lambda x: classify_with_regex(x) is None)
llm_indices = df.index[llm_needed_mask].tolist()

llm_texts = df.loc[llm_needed_mask, "Details"].tolist()

if llm_texts:
    llm_categories = classify_transactions_batch(llm_texts)

    # Ensure exact length match
    if len(llm_categories) != len(llm_indices):
        llm_categories = llm_categories[:len(llm_indices)]
        llm_categories += ["Unclassified"] * (len(llm_indices) - len(llm_categories))

    # Assign row by row
    for idx, cat in zip(llm_indices, llm_categories):
        df.at[idx, "Subcategory"] = cat
        df.at[idx, "Method"] = "llm"

# Assigning regex classifications for the rest
df.loc[~llm_needed_mask, "Subcategory"] = df.loc[~llm_needed_mask, "Details"].apply(classify_with_regex)
df.loc[~llm_needed_mask, "Method"] = "regex"

In [12]:
# Split subcategory into parent + child
def split_subcategory(subcat):
    if pd.isna(subcat):
        return ("Unclassified", "Unclassified")
    
    match = re.match(r"(.+?)\s*\((.+)\)", subcat)
    if match:
        return match.group(1).strip(), match.group(2).strip()
    else:
        return subcat.strip(), subcat.strip()

df[["MainCategory", "SubCategoryDetail"]] = df["Subcategory"].apply(
    lambda x: pd.Series(split_subcategory(x))
)

In [13]:
# Saving the results
classified_transactions = BASE_DIR / "resources" / "mpesa_classified.csv"

df.to_csv(classified_transactions, index=False)

## Insights

In [ ]:
df["Subcategory"].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.countplot(x="Subcategory", data=df, order=df["Subcategory"].value_counts().index)
plt.xticks(rotation=45)
plt.show()